# Notebook 03 - Yelp SVD Baseline Model

This notebook implements a matrix factorization SVD baseline model using the `surprise` library. The MF SVD model works by training on the temporal train split, fine-tuning on validation splits, and evaluating on the test splits using our ranking metrics.

In [3]:
!pip install "numpy<2" scikit-surprise

In [4]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
from surprise import SVD, Dataset, Reader, accuracy
import os

from google.colab import drive
drive.mount('/content/drive')
origin_dir = '/content/drive/MyDrive/rec_system/newV2'
results_dir = f"{origin_dir}/resultsV2"
os.makedirs(results_dir, exist_ok=True)

SEED = 42 # set pseudorandom seed for reproducability
np.random.seed(SEED)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# load training, validation, and test data

train_df = pd.read_parquet(f"{origin_dir}/train_reviews.parquet")
val_df   = pd.read_parquet(f"{origin_dir}/val_reviews.parquet")
test_df  = pd.read_parquet(f"{origin_dir}/test_reviews.parquet")

print(f"Train: {len(train_df)} reviews")
print(f"Val: {len(val_df)} reviews")
print(f"Test: {len(test_df)} reviews")

Train: 1990604 reviews
Val: 166488 reviews
Test: 166488 reviews


In [6]:
# pre-SVD training
# rating range given to reader for normalization,
# then converting train df to surprise dataset format
# to build trainset object for fitting

reader = Reader(rating_scale=(1, 5))
train_data = Dataset.load_from_df(train_df[["user_id", "business_id", "stars"]], reader)
trainset = train_data.build_full_trainset()
print(f"Trainset: {trainset.n_users} users, {trainset.n_items} businesses")

Trainset: 166488 users, 39111 businesses


In [7]:
# build SVD model for training and set hyperparameters
# latent space size 128-dim vectors to macth user + item vectors

svd = SVD(n_factors=128,
          n_epochs=20,
          lr_all=0.005,
          reg_all=0.02,
          random_state=42,
          verbose=True)

svd.fit(trainset)
print("Training completed.")

Processing epoch 0
Processing epoch 1
Processing epoch 2
Processing epoch 3
Processing epoch 4
Processing epoch 5
Processing epoch 6
Processing epoch 7
Processing epoch 8
Processing epoch 9
Processing epoch 10
Processing epoch 11
Processing epoch 12
Processing epoch 13
Processing epoch 14
Processing epoch 15
Processing epoch 16
Processing epoch 17
Processing epoch 18
Processing epoch 19
Training completed.


In [8]:
def compute_rmse(model, df):
  testset = list(zip(df["user_id"], df["business_id"], df["stars"]))
  preds = model.test(testset)
  return accuracy.rmse(preds, verbose=False)

# compute RMSE metrics for validation
val_rmse  = compute_rmse(svd, val_df)
print(f"Val RMSE:  {val_rmse:.4f}")

Val RMSE:  1.1957


In [9]:
all_business_ids = list(train_df["business_id"].unique())

def evaluate_sampled(model, eval_df, train_df, split_name, n_negatives=99, ks=[5, 10, 20], seed=SEED):
  np.random.seed(seed)
  user_train_items = train_df.groupby("user_id")["business_id"].apply(set).to_dict()
  results = {k: {"ndcg": [], "precision": [], "recall": []} for k in ks}

  eval_pos = eval_df[eval_df["label"] == 1].reset_index(drop=True)
  print(f"{split_name}: evaluating {len(eval_pos):,} positive interactions")

  for _, row in tqdm(eval_pos.iterrows(), total=len(eval_pos), desc=split_name):
    user_id = row["user_id"]
    pos_item = row["business_id"]
    seen = user_train_items.get(user_id, set())
    pool = [b for b in np.random.choice(all_business_ids, n_negatives * 3, replace=False)
            if b not in seen and b != pos_item]
    negatives = pool[:n_negatives]

    if len(negatives) < n_negatives:
      continue

    candidates = (
      [(pos_item, model.predict(user_id, pos_item).est)]
      + [(neg, model.predict(user_id, neg).est) for neg in negatives]
    )

    candidates.sort(key=lambda x: x[1], reverse=True)
    ranked_items = [item for item, _ in candidates]

    for k in ks:
      top_k = ranked_items[:k]
      hit = pos_item in top_k
      rank = ranked_items.index(pos_item) + 1

      results[k]["precision"].append(1.0 / k if hit else 0)
      results[k]["recall"].append(1.0 if hit else 0)
      results[k]["ndcg"].append(1.0 / np.log2(rank + 1) if hit else 0.0)

  rows = []
  for k in ks:
    rows.append({
      "K":           k,
      "NDCG@K":      np.mean(results[k]["ndcg"]),
      "Precision@K": np.mean(results[k]["precision"]),
      "Recall@K":    np.mean(results[k]["recall"]),
    })

  metrics_df = pd.DataFrame(rows).set_index("K")
  print(f"\n{split_name} Results:")
  print(metrics_df.round(4).to_string())
  return metrics_df

val_sampled_metrics  = evaluate_sampled(svd, val_df,  train_df, "Validation")
test_sampled_metrics = evaluate_sampled(svd, test_df, train_df, "Test")

Validation: evaluating 117,793 positive interactions


Validation:   0%|          | 0/117793 [00:00<?, ?it/s]


Validation Results:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0789       0.0250    0.1249
10  0.1101       0.0222    0.2223
20  0.1501       0.0191    0.3823
Test: evaluating 116,694 positive interactions


Test:   0%|          | 0/116694 [00:00<?, ?it/s]


Test Results:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0805       0.0256    0.1281
10  0.1123       0.0228    0.2275
20  0.1528       0.0194    0.3888


In [10]:
def evaluate_full_catalogue(model, eval_df, train_df, split_name, ks=[5, 10, 20]):
  ts = model.trainset
  qlat_mat = model.qi
  item_bi = model.bi
  glb_mu = ts.global_mean

  raw_to_inner_iid = {ts.to_raw_iid(i): i for i in range(ts.n_items)}
  user_train_items = train_df.groupby("user_id")["business_id"].apply(set).to_dict()

  eval_pos = eval_df[
    (eval_df["label"] == 1) &
    eval_df["business_id"].isin(raw_to_inner_iid)
  ].reset_index(drop=True)

  n_skipped = len(eval_df[eval_df["label"] == 1]) - len(eval_pos)
  print(f"{split_name}: {len(eval_pos):,} evaluable positives ({n_skipped:,} skipped; target item not in trainset)")

  results = {k: {"ndcg": [], "precision": [], "recall": []} for k in ks}

  for _, row in tqdm(eval_pos.iterrows(), total=len(eval_pos), desc=split_name):
    user_id    = row["user_id"]
    target_bid = row["business_id"]

    try:
      inner_uid = ts.to_inner_uid(user_id)
    except ValueError:
      continue

    usr_vec = model.pu[inner_uid]
    usr_bi = model.bu[inner_uid]

    scores = glb_mu + usr_bi + item_bi + qlat_mat @ usr_vec

    seen_raw = user_train_items.get(user_id, set())
    seen_inner = [raw_to_inner_iid[b] for b in seen_raw if b in raw_to_inner_iid]
    scores[seen_inner] = -np.inf

    target_inner = raw_to_inner_iid[target_bid]
    target_score = scores[target_inner]
    rank = int((scores > target_score).sum()) + 1

    for k in ks:
      hit = rank <= k
      results[k]["ndcg"].append(1.0 / np.log2(rank + 1) if hit else 0.0)
      results[k]["precision"].append(1.0 / k if hit else 0.0)
      results[k]["recall"].append(float(hit))

  rows = []
  for k in ks:
    rows.append({
      "K": k,
      "NDCG@K": np.mean(results[k]["ndcg"]),
      "Precision@K": np.mean(results[k]["precision"]),
      "Recall@K": np.mean(results[k]["recall"]),
    })

  metrics_df = pd.DataFrame(rows).set_index("K")
  print(f"\n{split_name} Results:")
  print(metrics_df.round(4).to_string())
  return metrics_df

val_metrics  = evaluate_full_catalogue(svd, val_df,  train_df, "Validation")
test_metrics = evaluate_full_catalogue(svd, test_df, train_df, "Test")

Validation: 117,760 evaluable positives (33 skipped; target item not in trainset)


Validation:   0%|          | 0/117760 [00:00<?, ?it/s]


Validation Results:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0003       0.0001    0.0004
10  0.0004       0.0001    0.0007
20  0.0005       0.0001    0.0013
Test: 116,592 evaluable positives (102 skipped; target item not in trainset)


Test:   0%|          | 0/116592 [00:00<?, ?it/s]


Test Results:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0002       0.0001    0.0004
10  0.0004       0.0001    0.0008
20  0.0005       0.0001    0.0016


In [11]:
val_sampled_metrics.to_csv(f"{results_dir}/svd_val_metrics_k100.csv")
test_sampled_metrics.to_csv(f"{results_dir}/svd_test_metrics_k100.csv")
val_metrics.to_csv(f"{results_dir}/svd_val_metrics_fullcat.csv")
test_metrics.to_csv(f"{results_dir}/svd_test_metrics_fullcat.csv")

print("Saved:")
print("svd_val_metrics_k100.csv")
print("svd_test_metrics_k100.csv")
print("svd_val_metrics_fullcat.csv")
print("svd_test_metrics_fullcat.csv")

Saved:
svd_val_metrics_k100.csv
svd_test_metrics_k100.csv
svd_val_metrics_fullcat.csv
svd_test_metrics_fullcat.csv


In [12]:
print("=" * 50)
print("SAMPLED EVALUATION (1 pos + 99 negatives)")
print("=" * 50)
print("\nValidation:")
print(val_sampled_metrics.round(4).to_string())
print("\nTest:")
print(test_sampled_metrics.round(4).to_string())

print("\n" + "=" * 50)
print("FULL-CATALOGUE EVALUATION (all 39k items)")
print("=" * 50)
print("\nValidation:")
print(val_metrics.round(4).to_string())
print("\nTest:")
print(test_metrics.round(4).to_string())

SAMPLED EVALUATION (1 pos + 99 negatives)

Validation:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0789       0.0250    0.1249
10  0.1101       0.0222    0.2223
20  0.1501       0.0191    0.3823

Test:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0805       0.0256    0.1281
10  0.1123       0.0228    0.2275
20  0.1528       0.0194    0.3888

FULL-CATALOGUE EVALUATION (all 39k items)

Validation:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0003       0.0001    0.0004
10  0.0004       0.0001    0.0007
20  0.0005       0.0001    0.0013

Test:
    NDCG@K  Precision@K  Recall@K
K                                
5   0.0002       0.0001    0.0004
10  0.0004       0.0001    0.0008
20  0.0005       0.0001    0.0016
